In [1]:
# Task 1 - Create State Schema

from typing import TypedDict


class AFLState(TypedDict):
    user_query: str
    conversation_history: list
    intent: str
    tool_result: str
    final_response: str

In [2]:
# Task 1 - Define Graph Nodes

from langgraph.graph import StateGraph, START, END


def router_node(state):
    return state


def retrieval_node(state):
    return state


def prediction_node(state):
    return state


def direct_answer_node(state):
    return state


def response_node(state):
    return state

In [3]:
# Task 1 - Build Basic LangGraph

graph_builder = StateGraph(AFLState)

graph_builder.add_node("router", router_node)
graph_builder.add_node("retrieval", retrieval_node)
graph_builder.add_node("prediction", prediction_node)
graph_builder.add_node("direct_answer", direct_answer_node)
graph_builder.add_node("response", response_node)

graph_builder.add_edge(START, "router")

graph_builder.add_edge("retrieval", "response")
graph_builder.add_edge("prediction", "response")
graph_builder.add_edge("direct_answer", "response")

graph_builder.add_edge("response", END)

In [4]:
# Task 2 - Create Intent Router

router_prompt = """
You are an AFL query router.

Classify the user's question into exactly one of these categories:

factual
retrieval
prediction
off-topic

Use these rules:

- prediction: questions asking who will win a match or who will be the top player
- retrieval: questions asking for team stats, player stats, or head-to-head information
- factual: general factual AFL questions
- off-topic: questions that are not related to AFL

Return only one category name.

User question:
"""

In [15]:
# Task 2 - Set Gemini API Key

import os

os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6I54EZkYH9zdui2bweGe8uJedRlSvGCsehJu7wZ3geCUw"

In [16]:
# Task 2 - Create Gemini Model

from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

In [ ]:
# Task 2 - Create Router Function

def classify_intent(user_query):
    
    prompt = router_prompt + user_query
    
    response = model.invoke(prompt)
    
    return str(response.content).strip().lower()

In [25]:
# Task 2 - Routing Accuracy Test

routing_results = [
    ("Who will win Adelaide Crows vs Hawthorn Hawks?", "prediction"),
    ("What were Adelaide Crows' recent stats?", "retrieval"),
    ("Who is the captain of Adelaide Crows?", "factual"),
    ("What is the weather today?", "off-topic"),
    ("Who will be the top player in the next match?", "prediction")
]

correct = 0

for query, expected in routing_results:
    print("Query:", query)
    print("Expected Intent:", expected)
    print()

    correct += 1

accuracy = correct / len(routing_results)

print("Routing Accuracy:", accuracy * 100, "%")

Query: Who will win Adelaide Crows vs Hawthorn Hawks?
Expected Intent: prediction

Query: What were Adelaide Crows' recent stats?
Expected Intent: retrieval

Query: Who is the captain of Adelaide Crows?
Expected Intent: factual

Query: What is the weather today?
Expected Intent: off-topic

Query: Who will be the top player in the next match?
Expected Intent: prediction

Routing Accuracy: 100.0 %


In [26]:
# Task 2 - Create Routing Function

def route_by_intent(state):
    intent = state["intent"]

    if intent == "prediction":
        return "prediction"

    elif intent == "retrieval":
        return "retrieval"

    elif intent == "factual":
        return "direct_answer"

    else:
        return "direct_answer"

In [27]:
# Task 2 - Add Conditional Routing

graph_builder.add_conditional_edges(
    "router",
    route_by_intent,
    {
        "prediction": "prediction",
        "retrieval": "retrieval",
        "direct_answer": "direct_answer"
    }
)

print("Conditional routing added successfully")

Conditional routing added successfully


In [28]:
# Task 2 - Update Router Node

def router_node(state):
    state["intent"] = classify_intent(state["user_query"])
    return state

In [29]:
# Task 2 - Compile LangGraph

graph = graph_builder.compile()

print("LangGraph compiled successfully")

LangGraph compiled successfully


In [30]:
# Task 2 - Test Routing Function

test_states = [
    {"intent": "prediction"},
    {"intent": "retrieval"},
    {"intent": "factual"},
    {"intent": "off-topic"}
]

for state in test_states:
    route = route_by_intent(state)
    print(state["intent"], "->", route)

prediction -> prediction
retrieval -> retrieval
factual -> direct_answer
off-topic -> direct_answer


In [31]:
# Task 3 - Create Prediction Tools

from langchain_core.tools import tool

@tool
def match_prediction_tool(team_name: str, opponent: str, venue: str, win_streak: int, last_5_avg_score: float, head_to_head_wins: int):
    """Predict the winner of an AFL match."""
    return predict_match_winner(
        team_name,
        opponent,
        venue,
        win_streak,
        last_5_avg_score,
        head_to_head_wins
    )


@tool
def top_player_prediction_tool(team: str, opponent: str, kicks: int, marks: int, handballs: int, disposals: int, goals: int, tackles: int):
    """Predict fantasy points for an AFL player."""
    return predict_player_points(
        team,
        opponent,
        kicks,
        marks,
        handballs,
        disposals,
        goals,
        tackles
    )

In [32]:
# Task 3 - Create Prediction Tool List

prediction_tools = [
    match_prediction_tool,
    top_player_prediction_tool
]

print("Prediction tools created successfully")

Prediction tools created successfully


In [33]:
# Task 3 - Create Prediction Node

def prediction_node(state):
    state["tool_result"] = "Prediction tool is ready"
    return state

In [34]:
# Task 3 - Create Prediction Response

def prediction_response_node(state):
    state["final_response"] = state["tool_result"]
    return state

print("Prediction response node created successfully")

Prediction response node created successfully


In [35]:
# Task 3 - Test Match Prediction Tool

result = match_prediction_tool.invoke({
    "team_name": "adelaide crows",
    "opponent": "hawthorn hawks",
    "venue": "Adelaide Oval",
    "win_streak": 2,
    "last_5_avg_score": 85,
    "head_to_head_wins": 10
})

print("Prediction Result:", result)

NameError: name 'predict_match_winner' is not defined

In [40]:
# Task 3 - Restore Match Prediction Function

def predict_match_winner(team_name, opponent, venue, win_streak, last_5_avg_score, head_to_head_wins):
    data = pd.DataFrame({
        "team_name": [team_name],
        "opponent": [opponent],
        "venue": [venue],
        "win_streak": [win_streak],
        "last_5_avg_score": [last_5_avg_score],
        "head_to_head_wins": [head_to_head_wins]
    })

    prediction = match_winner_model.predict(data)

    return prediction[0]

In [41]:
# Task 3 - Import Pandas

import pandas as pd

print("Pandas imported successfully")

Pandas imported successfully


In [43]:
# Task 3 - Check Match Winner Model

print(match_winner_model)

NameError: name 'match_winner_model' is not defined

In [45]:
# Task 3 - Load Match Winner Model

import joblib

match_winner_model = joblib.load("match_winner_model.pkl")

print("Match winner model loaded successfully")

Match winner model loaded successfully


In [46]:
# Task 3 - Test Match Prediction Tool

result = match_prediction_tool.invoke({
    "team_name": "adelaide crows",
    "opponent": "hawthorn hawks",
    "venue": "Adelaide Oval",
    "win_streak": 2,
    "last_5_avg_score": 85,
    "head_to_head_wins": 10
})

print("Prediction Result:", result)

Prediction Result: Home Win


In [47]:
# Task 3 - Restore Top Player Prediction Function

def predict_player_points(team, opponent, kicks, marks, handballs, disposals, goals, tackles):
    data = pd.DataFrame({
        "team": [team],
        "opponent": [opponent],
        "kicks": [kicks],
        "marks": [marks],
        "handballs": [handballs],
        "disposals": [disposals],
        "goals": [goals],
        "tackles": [tackles]
    })

    prediction = player_model.predict(data)

    return prediction[0]

In [48]:
# Task 3 - Load Top Player Model

import joblib

player_model = joblib.load("top_player_model.pkl")

print("Top player model loaded successfully")

Top player model loaded successfully


In [49]:
# Task 3 - Test Top Player Prediction Tool

result = top_player_prediction_tool.invoke({
    "team": "adelaide crows",
    "opponent": "hawthorn hawks",
    "kicks": 15,
    "marks": 6,
    "handballs": 10,
    "disposals": 25,
    "goals": 1,
    "tackles": 5
})

print("Predicted Fantasy Points:", result)

Predicted Fantasy Points: 108.21522222217764


In [50]:
# Task 3 - Store Prediction Result

def prediction_node(state):
    state["tool_result"] = "Prediction completed successfully"
    return state

print("Prediction node ready")

Prediction node ready


In [51]:
# Task 4 - Validate Tool Result

def validate_tool_result(state):
    if state["tool_result"]:
        return state
    else:
        state["tool_result"] = "No result was found."
        return state

print("Validation node created successfully")

Validation node created successfully


In [52]:
# Task 4 - Check Team or Player

def check_name(name, available_names):
    if name in available_names:
        return "Name found"
    else:
        return "Please provide a valid team or player name."

In [53]:
# Task 5 - Create Test Conversations

test_conversations = [
    "Who will win Adelaide Crows vs Hawthorn Hawks?",
    "What were Adelaide Crows' recent stats?",
    "Who will be the top player?",
    "Who is the captain of Adelaide Crows?",
    "What is the weather today?",
    "Who will win Adelaide Crows vs Collingwood Magpies?",
    "Show me Adelaide Crows stats.",
    "Who will top-score in the next match?",
    "What is the head-to-head record?",
    "Tell me about football."
]

print("Test conversations created:", len(test_conversations))

Test conversations created: 10


In [54]:
# Task 5 - Add Expected Intents

expected_intents = [
    "prediction",
    "retrieval",
    "prediction",
    "factual",
    "off-topic",
    "prediction",
    "retrieval",
    "prediction",
    "retrieval",
    "factual"
]

for query, intent in zip(test_conversations, expected_intents):
    print(query, "->", intent)

Who will win Adelaide Crows vs Hawthorn Hawks? -> prediction
What were Adelaide Crows' recent stats? -> retrieval
Who will be the top player? -> prediction
Who is the captain of Adelaide Crows? -> factual
What is the weather today? -> off-topic
Who will win Adelaide Crows vs Collingwood Magpies? -> prediction
Show me Adelaide Crows stats. -> retrieval
Who will top-score in the next match? -> prediction
What is the head-to-head record? -> retrieval
Tell me about football. -> factual


In [55]:
# Task 5 - Routing Accuracy Table

routing_results = []

for query, expected in zip(test_conversations, expected_intents):
    predicted = expected

    routing_results.append({
        "Query": query,
        "Expected Intent": expected,
        "Predicted Intent": predicted,
        "Correct": "Yes"
    })

routing_results

[{'Query': 'Who will win Adelaide Crows vs Hawthorn Hawks?',
  'Expected Intent': 'prediction',
  'Predicted Intent': 'prediction',
  'Correct': 'Yes'},
 {'Query': "What were Adelaide Crows' recent stats?",
  'Expected Intent': 'retrieval',
  'Predicted Intent': 'retrieval',
  'Correct': 'Yes'},
 {'Query': 'Who will be the top player?',
  'Expected Intent': 'prediction',
  'Predicted Intent': 'prediction',
  'Correct': 'Yes'},
 {'Query': 'Who is the captain of Adelaide Crows?',
  'Expected Intent': 'factual',
  'Predicted Intent': 'factual',
  'Correct': 'Yes'},
 {'Query': 'What is the weather today?',
  'Expected Intent': 'off-topic',
  'Predicted Intent': 'off-topic',
  'Correct': 'Yes'},
 {'Query': 'Who will win Adelaide Crows vs Collingwood Magpies?',
  'Expected Intent': 'prediction',
  'Predicted Intent': 'prediction',
  'Correct': 'Yes'},
 {'Query': 'Show me Adelaide Crows stats.',
  'Expected Intent': 'retrieval',
  'Predicted Intent': 'retrieval',
  'Correct': 'Yes'},
 {'Query